# Module 10 — Dynamic Programming 1D and Sequence Patterns

## What you will discover

Every cell below runs this module's **real** problem-bank solutions and asserts
their behaviour. Nothing here prints a claim it has not computed.

The assertions are lifted directly from `problems/tests/`, so they cannot drift
from the implementations — if a signature changes, the tests break first.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

The solutions directory goes on `sys.path` relative to this notebook's own
location. Never hard-code an absolute path — `tools/check_links.py` fails the
build on them, because a path with a username in it works on exactly one
machine.

In [ ]:
import sys
import time
from pathlib import Path

import pytest   # some assertions check that an invalid input RAISES
sys.path.insert(0, str(Path.cwd() / "problems" / "solutions"))

from p01_climb_stairs import climb_stairs
from p02_house_robber import house_robber
from p03_coin_change_min import coin_change_min

print("module 10: Dynamic Programming 1D and Sequence Patterns")
print("problems available:", 8)
for name in ['p01_climb_stairs', 'p02_house_robber', 'p03_coin_change_min', 'p04_lis', 'p05_word_break', 'p06_decode_ways', 'p07_max_product_subarray', 'p08_stock_with_cooldown']:
    print(f"  {name}")

## 1. Baseline — `p01_climb_stairs`

The first property, asserted rather than printed. Read the assertions before
running: each one names a specific input class, and most cross-check against an
independent brute force over the same data.

In [ ]:
assert climb_stairs(0) == 1
assert climb_stairs(1) == 1
assert climb_stairs(2) == 2
assert climb_stairs(3) == 3
assert climb_stairs(4) == 5
assert climb_stairs(10) == 89
with pytest.raises(ValueError):
    climb_stairs(-1)
# Cross-check against an exhaustive enumeration for small n.
def brute(k):
    if k < 0:
        return 0
    if k == 0:
        return 1
    return brute(k - 1) + brute(k - 2)
for k in range(0, 16):
    assert climb_stairs(k) == brute(k), k
# Scale: O(2^n) recursion would never finish this.
assert climb_stairs(100_000) > 0

print("all assertions held")

## 2. Predict before you run

For maximum subarray *sum* a single running best suffices. Predict whether the same is true for maximum subarray *product* on `[-2, 3, -4]`, and if not, write down what the second piece of state must be.

Commit to an answer before executing the next cell. Predicting and being wrong
is what makes the correction stick; reading the output first does not.

In [ ]:
assert house_robber([1, 2, 3, 1]) == 4
assert house_robber([2, 7, 9, 3, 1]) == 12
assert house_robber([]) == 0
assert house_robber([5]) == 5
# Two houses: take the larger.
assert house_robber([2, 1]) == 2
assert house_robber([1, 2]) == 2
# All zeros.
assert house_robber([0, 0, 0]) == 0
# Adjacent large values force a skip.
assert house_robber([100, 1, 1, 100]) == 200
# Cross-check against brute-force over all valid subsets.
import itertools
for data in ([1, 2, 3, 1], [2, 7, 9, 3, 1], [5, 1, 2, 6], [4, 4, 4, 4, 4]):
    best = 0
    for r in range(len(data) + 1):
        for combo in itertools.combinations(range(len(data)), r):
            if all(b - a > 1 for a, b in zip(combo, combo[1:])):
                best = max(best, sum(data[i] for i in combo))
    assert house_robber(data) == best, data

print("all assertions held")

## 3. Measurement

Claims about complexity are claims about wall-clock behaviour at scale, so they
have to be measured rather than asserted from the shape of the code.

In [ ]:
started = time.perf_counter()

assert coin_change_min([1, 2, 5], 11) == 3
assert coin_change_min([2], 3) == -1
assert coin_change_min([1], 0) == 0
# Zero amount always needs zero coins.
assert coin_change_min([5], 0) == 0
# The case greedy gets wrong: 3+3 beats 4+1+1.
assert coin_change_min([1, 3, 4], 6) == 2
# Exact single coin.
assert coin_change_min([1, 2, 5], 5) == 1
# Impossible because every coin is too large.
assert coin_change_min([7, 11], 5) == -1
# Only 1s available.
assert coin_change_min([1], 25) == 25
# Coin larger than int32 must not break anything.
assert coin_change_min([1, 2147483647], 3) == 3
# Cross-check against BFS over amounts.
from collections import deque
def brute(cs, amt):
    if amt == 0:
        return 0
    seen = {0}
    q = deque([(0, 0)])
    while q:
        total, steps = q.popleft()
        for c in cs:
            nt = total + c
            if nt == amt:
                return steps + 1
            if nt < amt and nt not in seen:
                seen.add(nt)
                q.append((nt, steps + 1))
    return -1
for cs, amt in (([1, 3, 4], 6), ([2, 5], 11), ([3, 7], 5), ([1, 5, 6, 9], 11)):
    assert coin_change_min(cs, amt) == brute(cs, amt), (cs, amt)

elapsed = (time.perf_counter() - started) * 1000
print(f"all assertions held in {elapsed:.2f} ms")

## 4. Fix this cell

The values below are **wrong on purpose**. Run it, read the failure, work out
the right numbers from the cells above, and correct them in place.

Change only the expected values — not the code that computes them.

In [ ]:
import os

# DELIBERATELY BROKEN - two expected values, both wrong. Fix in place.

expected_problem_count = 99      # how many problems does this module ship?
expected_solution_count = 99     # how many reference solutions are on disk?

problem_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems") if f.startswith("p") and f.endswith(".py")
)
solution_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems" / "solutions")
    if f.startswith("p") and f.endswith(".py")
)

assert expected_problem_count == len(problem_files), (
    f"expected {expected_problem_count} problems, found {len(problem_files)}"
)
assert expected_solution_count == len(solution_files), (
    f"expected {expected_solution_count} solutions, found {len(solution_files)}"
)
print("Both match. Every problem has exactly one reference solution.")

## Takeaways

1. State, transition, base case, order - in that order, on paper, before any code.
2. When a transition can reverse the ordering, one extreme is not enough state.
3. An 'impossible' sentinel must be a value the valid range can never take.

### Where to go next

- [`01_README.md`](01_README.md) — the concepts in depth
- [`problems/README.md`](problems/README.md) — all 8 problems, with hint ladders
- [`debug_lab/SYMPTOMS.md`](debug_lab/SYMPTOMS.md) — planted defects that exit 0
- [Pattern Recognition Guide](../PATTERN_RECOGNITION_GUIDE.md) — attacking an unseen problem